# MCL-WiC Results Consolidation

This notebook consolidates experiment outputs produced by `mcl_wic_colab_common.py`.

It generates:
- best hyperparameters selected for each backbone/variant;
- accuracy summaries by dataset with standard deviation across seeds;
- Critical Difference (CD) analyses overall, for monolingual, multilingual, and cross-lingual splits.

Expected input structure:
`Experiments/<Backbone>/Results/<variant>/{summary.json,optuna_trials.json,seed_*.json}`.

In [35]:
# !pip install autorank

In [36]:
from pathlib import Path
import json
import math
import os
import warnings

BASE_DIR = Path.cwd()
if BASE_DIR.name != 'Experiments':
    BASE_DIR = BASE_DIR / 'Experiments'

OUTPUT_DIR = BASE_DIR / 'analysis_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUTPUT_DIR / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('XDG_CACHE_HOME', str(CACHE_DIR))
MPLCONFIG_DIR = OUTPUT_DIR / 'mplconfig'
MPLCONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPLCONFIG_DIR))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

RESULT_ROOTS = [BASE_DIR / 'Bert-base' / 'Results', BASE_DIR / 'XLM-RoBERTa' / 'Results']
MAKE_CD_PLOTS = True  # Export PNG Critical Difference diagrams.

MONOLINGUAL_SPLITS = ['test.en-en']
MULTILINGUAL_SPLITS = ['test.ar-ar', 'test.fr-fr', 'test.ru-ru', 'test.zh-zh']
CROSSLINGUAL_SPLITS = ['test.en-ar', 'test.en-fr', 'test.en-ru', 'test.en-zh']
TEST_SPLITS = MONOLINGUAL_SPLITS + MULTILINGUAL_SPLITS + CROSSLINGUAL_SPLITS
ALL_SPLITS = ['dev.en-en'] + TEST_SPLITS

VARIANT_NAMES = {
    'cls': 'PairCLS',
    'marked': 'MarkCLS',
    'target': 'TargetSim',
    'target_pos': 'TargetPOS',
    'gat': 'GloSeGAT',
    'bert_cls': 'PairCLS',
    'bert_marked': 'MarkCLS',
    'glowic_target': 'TargetSim',
    'glowic_target_pos': 'TargetPOS',
    'glowic_gat': 'GloSeGAT',
}

BACKBONE_NAMES = {
    'Bert-base': 'mBERT',
    'XLM-RoBERTa': 'XLM-R',
}

def load_json(path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

def variant_label(result_dir, config=None):
    raw = result_dir.name
    if config and config.get('variant'):
        raw = config['variant']
    return VARIANT_NAMES.get(raw, VARIANT_NAMES.get(result_dir.name, raw))

def backbone_label(result_dir, config=None):
    root_name = result_dir.parent.parent.name
    if config:
        model_name = str(config.get('model_name', '')).lower()
        if 'xlm' in model_name or 'roberta' in model_name:
            return 'XLM-R'
        if 'bert' in model_name:
            return 'mBERT'
    return BACKBONE_NAMES.get(root_name, root_name)

def collect_seed_runs(result_dir, summary=None):
    if summary and isinstance(summary.get('seed_runs'), list) and summary['seed_runs']:
        return summary['seed_runs']
    runs = []
    for seed_path in sorted(result_dir.glob('seed_*.json')):
        payload = load_json(seed_path)
        run = payload.get('run')
        if isinstance(run, dict) and isinstance(run.get('final'), dict):
            runs.append(run)
    return runs

def load_best_hparams(result_dir, summary=None):
    if summary and isinstance(summary.get('best_hparams'), dict):
        return summary['best_hparams'], summary.get('best_optuna_dev_accuracy')
    optuna_path = result_dir / 'optuna_trials.json'
    if optuna_path.exists():
        payload = load_json(optuna_path)
        return payload.get('best_hparams', {}), payload.get('best_dev_accuracy')
    return {}, None

def aggregate_seed_runs(seed_runs):
    rows = []
    for run in seed_runs:
        seed = run.get('seed')
        for split, metrics in run.get('final', {}).items():
            row = {'seed': seed, 'split': split}
            row.update(metrics)
            rows.append(row)
    raw = pd.DataFrame(rows)
    if raw.empty:
        return raw, pd.DataFrame()
    metric_cols = [c for c in ['accuracy', 'precision', 'recall', 'f1', 'loss'] if c in raw.columns]
    agg = raw.groupby('split')[metric_cols].agg(['mean', 'std', 'count'])
    agg.columns = [f'{metric}_{stat}' for metric, stat in agg.columns]
    agg = agg.reset_index()
    return raw, agg

def discover_experiments():
    experiments = []
    seed_rows = []
    metric_rows = []
    hparam_rows = []
    for root in RESULT_ROOTS:
        if not root.exists():
            continue
        for result_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
            summary_path = result_dir / 'summary.json'
            optuna_path = result_dir / 'optuna_trials.json'
            seed_paths = sorted(result_dir.glob('seed_*.json'))
            if not summary_path.exists() and not optuna_path.exists() and not seed_paths:
                continue
            summary = load_json(summary_path) if summary_path.exists() else None
            config = (summary or {}).get('config')
            if config is None and optuna_path.exists():
                config = load_json(optuna_path).get('config', {})
            config = config or {}
            backbone = backbone_label(result_dir, config)
            variant = variant_label(result_dir, config)
            model = f'{backbone} ({variant})'
            seed_runs = collect_seed_runs(result_dir, summary)
            raw, agg = aggregate_seed_runs(seed_runs)
            best_hparams, best_dev = load_best_hparams(result_dir, summary)
            experiments.append({'model': model, 'backbone': backbone, 'variant': variant, 'path': str(result_dir), 'n_seed_runs': len(seed_runs), 'best_optuna_dev_accuracy': best_dev})
            for key, value in best_hparams.items():
                hparam_rows.append({'model': model, 'backbone': backbone, 'variant': variant, 'hyperparameter': key, 'value': value})
            if not raw.empty:
                raw.insert(0, 'model', model)
                raw.insert(1, 'backbone', backbone)
                raw.insert(2, 'variant', variant)
                seed_rows.append(raw)
            if not agg.empty:
                agg.insert(0, 'model', model)
                agg.insert(1, 'backbone', backbone)
                agg.insert(2, 'variant', variant)
                metric_rows.append(agg)
    experiments_df = pd.DataFrame(experiments).sort_values(['backbone', 'variant']).reset_index(drop=True)
    seed_df = pd.concat(seed_rows, ignore_index=True) if seed_rows else pd.DataFrame()
    metrics_df = pd.concat(metric_rows, ignore_index=True) if metric_rows else pd.DataFrame()
    hparams_long = pd.DataFrame(hparam_rows)
    return experiments_df, seed_df, metrics_df, hparams_long


## Load Experiment Outputs

In [37]:
experiments_df, seed_df, metrics_df, hparams_long = discover_experiments()

display(experiments_df)

experiments_df.to_csv(OUTPUT_DIR / 'experiment_inventory.csv', index=False)
seed_df.to_csv(OUTPUT_DIR / 'seed_level_metrics.csv', index=False)
metrics_df.to_csv(OUTPUT_DIR / 'split_metric_summary_long.csv', index=False)
hparams_long.to_csv(OUTPUT_DIR / 'best_hyperparameters_long.csv', index=False)

print(f'Loaded {len(experiments_df)} experiment directories.')
print(f'Seed-level rows: {len(seed_df)}')
print(f'Outputs written to: {OUTPUT_DIR}')

,model,backbone,variant,path,n_seed_runs,best_optuna_dev_accuracy
0,XLM-R (GloSeGAT),XLM-R,GloSeGAT,/Users/antonio/Desktop/repositorios/Experiment...,15,0.901
1,XLM-R (MarkCLS),XLM-R,MarkCLS,/Users/antonio/Desktop/repositorios/Experiment...,15,0.876
2,XLM-R (PairCLS),XLM-R,PairCLS,/Users/antonio/Desktop/repositorios/Experiment...,15,0.842
3,XLM-R (TargetPOS),XLM-R,TargetPOS,/Users/antonio/Desktop/repositorios/Experiment...,15,0.892
4,XLM-R (TargetSim),XLM-R,TargetSim,/Users/antonio/Desktop/repositorios/Experiment...,15,0.890
5,mBERT (GloSeGAT),mBERT,GloSeGAT,/Users/antonio/Desktop/repositorios/Experiment...,15,0.846
6,mBERT (MarkCLS),mBERT,MarkCLS,/Users/antonio/Desktop/repositorios/Experiment...,15,0.855
7,mBERT (PairCLS),mBERT,PairCLS,/Users/antonio/Desktop/repositorios/Experiment...,15,0.838
8,mBERT (TargetPOS),mBERT,TargetPOS,/Users/antonio/Desktop/repositorios/Experiment...,15,0.839
9,mBERT (TargetSim),mBERT,TargetSim,/Users/antonio/Desktop/repositorios/Experiment...,15,0.850


Loaded 10 experiment directories.
Seed-level rows: 1500
Outputs written to: /Users/antonio/Desktop/repositorios/Experiments/analysis_outputs


## Best Hyperparameters by Backbone and Variant

In [38]:
if hparams_long.empty:
    print('No hyperparameters found.')
else:
    hparams_table = hparams_long.pivot_table(index=['backbone', 'variant', 'model'], columns='hyperparameter', values='value', aggfunc='first').reset_index()
    hparams_table = hparams_table.merge(experiments_df[['model', 'n_seed_runs', 'best_optuna_dev_accuracy']], on='model', how='left')
    display(hparams_table)
    hparams_table.to_csv(OUTPUT_DIR / 'best_hyperparameters_wide.csv', index=False)


,backbone,variant,model,batch_size,dropout,lr_encoder,lr_head,topk,warmup_ratio,weight_decay,n_seed_runs,best_optuna_dev_accuracy
0,XLM-R,GloSeGAT,XLM-R (GloSeGAT),8.0,0.077899,0.000007,0.000027,1.0,0.078518,0.030593,15,0.901
1,XLM-R,MarkCLS,XLM-R (MarkCLS),8.0,0.078185,0.000006,0.000026,NaN,0.098689,0.027135,15,0.876
2,XLM-R,PairCLS,XLM-R (PairCLS),4.0,0.195801,0.000003,0.000160,NaN,0.011587,0.035343,15,0.842
3,XLM-R,TargetPOS,XLM-R (TargetPOS),2.0,0.108246,0.000005,0.000060,NaN,0.036636,0.021597,15,0.892
4,XLM-R,TargetSim,XLM-R (TargetSim),4.0,0.163155,0.000006,0.000053,NaN,0.094581,0.031344,15,0.890
5,mBERT,GloSeGAT,mBERT (GloSeGAT),8.0,0.108795,0.000026,0.000418,7.0,0.040702,0.008849,15,0.846
6,mBERT,MarkCLS,mBERT (MarkCLS),8.0,0.147599,0.000012,0.000079,NaN,0.124311,0.004523,15,0.855
7,mBERT,PairCLS,mBERT (PairCLS),16.0,0.248757,0.000011,0.000406,NaN,0.082007,0.025878,15,0.838
8,mBERT,TargetPOS,mBERT (TargetPOS),8.0,0.292519,0.000011,0.000444,NaN,0.066023,0.096563,15,0.839
9,mBERT,TargetSim,mBERT (TargetSim),4.0,0.322796,0.000012,0.000156,NaN,0.078010,0.003439,15,0.850


## Accuracy Summary by Dataset

The table below reports mean accuracy and standard deviation across seed runs. It includes the development split and all test splits.

In [39]:
def split_type(split):
    if split in MONOLINGUAL_SPLITS:
        return 'monolingual'
    if split in MULTILINGUAL_SPLITS:
        return 'multilingual'
    if split in CROSSLINGUAL_SPLITS:
        return 'cross-lingual'
    if split == 'dev.en-en':
        return 'development'
    return 'other'

def format_mean_std_pct(mean, std, digits=2, bold=False):
    if pd.isna(mean):
        return ''
    mean_pct = 100 * mean
    if pd.isna(std):
        value = f'{mean_pct:.{digits}f}'
    else:
        value = f'{mean_pct:.{digits}f} ± {100 * std:.{digits}f}'
    return f'\\textbf{{{value}}}' if bold else value

acc_summary = metrics_df[['model', 'backbone', 'variant', 'split', 'accuracy_mean', 'accuracy_std', 'accuracy_count']].copy()
acc_summary = acc_summary[acc_summary['split'].isin(TEST_SPLITS)].copy()  # article table uses test datasets only
acc_summary['dataset_type'] = acc_summary['split'].map(split_type)
acc_summary['accuracy_mean_pct'] = 100 * acc_summary['accuracy_mean']
acc_summary['accuracy_std_pct'] = 100 * acc_summary['accuracy_std']
VARIANT_ORDER = ['PairCLS', 'MarkCLS', 'TargetSim', 'TargetPOS', 'GloSeGAT']
BACKBONE_ORDER = ['mBERT', 'XLM-R']
DATASET_TYPE_ORDER = ['monolingual', 'multilingual', 'cross-lingual']
acc_summary['split'] = pd.Categorical(acc_summary['split'], categories=TEST_SPLITS, ordered=True)
acc_summary['variant'] = pd.Categorical(acc_summary['variant'], categories=VARIANT_ORDER, ordered=True)
acc_summary['backbone'] = pd.Categorical(acc_summary['backbone'], categories=BACKBONE_ORDER, ordered=True)
acc_summary['dataset_type'] = pd.Categorical(acc_summary['dataset_type'], categories=DATASET_TYPE_ORDER, ordered=True)
acc_summary = acc_summary.sort_values(['variant', 'backbone', 'split'])

# Numeric table: models are rows; dataset_type and split define the dataset columns.
accuracy_numeric = acc_summary.pivot_table(
    index=['backbone', 'variant', 'model'],
    columns=['dataset_type', 'split'],
    values='accuracy_mean_pct',
    aggfunc='first',
)

accuracy_std_numeric = acc_summary.pivot_table(
    index=['backbone', 'variant', 'model'],
    columns=['dataset_type', 'split'],
    values='accuracy_std_pct',
    aggfunc='first',
)

# Preserve the intended split order inside the two-level dataset header.
ordered_columns = []
for split in TEST_SPLITS:
    col = (split_type(split), split)
    if col in accuracy_numeric.columns:
        ordered_columns.append(col)
accuracy_numeric = accuracy_numeric.reindex(columns=pd.MultiIndex.from_tuples(ordered_columns, names=['dataset_type', 'split']))
accuracy_std_numeric = accuracy_std_numeric.reindex(columns=accuracy_numeric.columns)

# Formatted table: bold the best model for each dataset column within each backbone separately.
accuracy_table = accuracy_numeric.copy().astype(object)
for col in accuracy_numeric.columns:
    best_by_backbone = accuracy_numeric[col].groupby(level='backbone').transform('max')
    for idx in accuracy_numeric.index:
        mean = accuracy_numeric.loc[idx, col]
        std = accuracy_std_numeric.loc[idx, col]
        best = bool(mean == best_by_backbone.loc[idx])
        accuracy_table.loc[idx, col] = format_mean_std_pct(mean / 100, std / 100 if not pd.isna(std) else np.nan, bold=best)

accuracy_table = accuracy_table.reset_index()
accuracy_numeric_out = accuracy_numeric.reset_index()
accuracy_std_numeric_out = accuracy_std_numeric.reset_index()

# Long-form table is convenient for filtering and plotting.
accuracy_long_pct = acc_summary[['dataset_type', 'split', 'backbone', 'variant', 'model', 'accuracy_mean_pct', 'accuracy_std_pct', 'accuracy_count']].copy()

display(accuracy_table)

accuracy_table.to_csv(OUTPUT_DIR / 'accuracy_by_dataset_mean_std.csv', index=False)
accuracy_numeric_out.to_csv(OUTPUT_DIR / 'accuracy_by_dataset_numeric.csv', index=False)
accuracy_std_numeric_out.to_csv(OUTPUT_DIR / 'accuracy_by_dataset_std_numeric.csv', index=False)
accuracy_long_pct.to_csv(OUTPUT_DIR / 'accuracy_by_dataset_long_pct.csv', index=False)

dataset_type backbone    variant              model            monolingual           multilingual                                                                               cross-lingual  \
split                                                           test.en-en             test.ar-ar             test.fr-fr             test.ru-ru             test.zh-zh             test.en-ar   
0               mBERT    PairCLS    mBERT (PairCLS)           81.34 ± 0.67           73.89 ± 0.61           76.53 ± 1.22           70.85 ± 0.96           69.84 ± 1.22           55.51 ± 1.73   
1               mBERT    MarkCLS    mBERT (MarkCLS)           83.77 ± 1.04  \textbf{75.10 ± 0.90}           79.09 ± 0.64           73.23 ± 0.74           74.34 ± 2.10           59.52 ± 2.16   
2               mBERT  TargetSim  mBERT (TargetSim)  \textbf{85.15 ± 0.93}           72.73 ± 1.79  \textbf{80.51 ± 1.13}           76.68 ± 0.79  \textbf{79.96 ± 0.92}           66.21 ± 4.42   
3               mBERT  TargetPOS  mBERT (TargetPOS)           84.65 ± 1.52           74.15 ± 1.21  \textbf{80.51 ± 1.04}  \textbf{77.20 ± 0.85}           79.67 ± 1.07           67.60 ± 2.62   
4               mBERT   GloSeGAT   mBERT (GloSeGAT)           84.89 ± 0.72           72.77 ± 1.13           78.18 ± 1.00           76.40 ± 0.92           78.53 ± 1.14  \textbf{72.11 ± 2.87}   
5               XLM-R    PairCLS    XLM-R (PairCLS)           81.46 ± 1.63           77.47 ± 1.06           77.51 ± 1.38           72.67 ± 1.59           76.29 ± 1.81           72.79 ± 2.35   
6               XLM-R    MarkCLS    XLM-R (MarkCLS)  \textbf{89.95 ± 0.73}  \textbf{82.21 ± 0.49}  \textbf{83.70 ± 0.61}  \textbf{82.41 ± 0.75}           73.41 ± 2.76  \textbf{82.32 ± 1.39}   
7               XLM-R  TargetSim  XLM-R (TargetSim)          87.53 ± 10.41           76.99 ± 7.59           82.61 ± 9.05           81.69 ± 8.80  \textbf{82.39 ± 9.00}           80.39 ± 8.76   
8               XLM-R  TargetPOS  XLM-R (TargetPOS)          68.65 ± 20.86          64.17 ± 15.43          66.65 ± 18.41          65.87 ± 17.88          66.09 ± 18.04          66.04 ± 18.12   
9               XLM-R   GloSeGAT   XLM-R (GloSeGAT)          87.74 ± 10.46           76.06 ± 7.38           82.10 ± 8.94           80.94 ± 8.64           81.21 ± 8.79           80.09 ± 8.71   

dataset_type                                                                        
split                    test.en-fr             test.en-ru              test.en-zh  
0                      65.90 ± 2.38           61.74 ± 2.06            55.31 ± 1.57  
1                      71.03 ± 2.37           67.00 ± 1.58            61.49 ± 3.04  
2                      80.20 ± 2.68           78.25 ± 2.67            80.79 ± 3.09  
3                      80.43 ± 1.18           78.40 ± 1.51            80.97 ± 2.27  
4             \textbf{82.47 ± 1.50}  \textbf{81.14 ± 1.29}   \textbf{84.47 ± 2.03}  
5                      70.53 ± 3.58           71.59 ± 3.41            69.36 ± 3.86  
6                      82.41 ± 1.87           82.97 ± 1.37            82.87 ± 1.78  
7             \textbf{84.84 ± 9.68}  \textbf{85.85 ± 9.96}  \textbf{86.78 ± 10.19}  
8                     67.55 ± 19.73          67.65 ± 19.90           68.61 ± 20.78  
9                      84.63 ± 9.68           85.23 ± 9.85           86.49 ± 10.24

## LaTeX Table for Article Drafting

In [40]:
latex_path = OUTPUT_DIR / 'accuracy_by_dataset_mean_std.tex'
latex_table = accuracy_table.to_latex(index=False, escape=False)
latex_path.write_text(latex_table, encoding='utf-8')
print(latex_table)
print(f'LaTeX table written to {latex_path}')

\begin{tabular}{llllllllllll}
\toprule
backbone & variant & model & monolingual & \multicolumn{4}{r}{multilingual} & \multicolumn{4}{r}{cross-lingual} \\
 &  &  & test.en-en & test.ar-ar & test.fr-fr & test.ru-ru & test.zh-zh & test.en-ar & test.en-fr & test.en-ru & test.en-zh \\
\midrule
mBERT & PairCLS & mBERT (PairCLS) & 81.34 ± 0.67 & 73.89 ± 0.61 & 76.53 ± 1.22 & 70.85 ± 0.96 & 69.84 ± 1.22 & 55.51 ± 1.73 & 65.90 ± 2.38 & 61.74 ± 2.06 & 55.31 ± 1.57 \\
mBERT & MarkCLS & mBERT (MarkCLS) & 83.77 ± 1.04 & \textbf{75.10 ± 0.90} & 79.09 ± 0.64 & 73.23 ± 0.74 & 74.34 ± 2.10 & 59.52 ± 2.16 & 71.03 ± 2.37 & 67.00 ± 1.58 & 61.49 ± 3.04 \\
mBERT & TargetSim & mBERT (TargetSim) & \textbf{85.15 ± 0.93} & 72.73 ± 1.79 & \textbf{80.51 ± 1.13} & 76.68 ± 0.79 & \textbf{79.96 ± 0.92} & 66.21 ± 4.42 & 80.20 ± 2.68 & 78.25 ± 2.67 & 80.79 ± 3.09 \\
mBERT & TargetPOS & mBERT (TargetPOS) & 84.65 ± 1.52 & 74.15 ± 1.21 & \textbf{80.51 ± 1.04} & \textbf{77.20 ± 0.85} & 79.67 ± 1.07 & 67.60 ± 2.62 & 80.43 

## Critical Difference Analysis with Autorank

This section uses `autorank` to run the statistical comparison and generate Critical Difference diagrams. If `autorank` is not installed, the notebook installs it automatically with `pip`.

The comparison matrix uses paired `split+seed` blocks as rows and model variants as columns. This keeps the analysis paired and also provides enough observations for subsets with few datasets, including the monolingual English test split. Accuracy values are expressed as percentages.


In [41]:
import sys
import subprocess

try:
    from autorank import autorank, plot_stats, create_report
except ModuleNotFoundError:
    print('Installing autorank...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'autorank'])
    from autorank import autorank, plot_stats, create_report


def autorank_compare_matrix(splits, backbone=None, metric='accuracy'):
    df = seed_df[seed_df['split'].isin(splits)].copy()
    if backbone is not None:
        df = df[df['backbone'] == backbone].copy()
    if df.empty:
        return pd.DataFrame()
    df['block'] = df['split'].astype(str) + '__seed_' + df['seed'].astype(str)
    df[f'{metric}_pct'] = 100 * df[metric]
    mat = df.pivot_table(index='block', columns='model', values=f'{metric}_pct', aggfunc='first')
    # Keep all selected model columns and retain only paired blocks complete for every selected model.
    mat = mat.dropna(axis=0, how='any')
    mat = mat.dropna(axis=1, how='all')
    return mat


def run_autorank_analysis(name, splits, backbone=None, alpha=0.05):
    df_compare = autorank_compare_matrix(splits, backbone=backbone, metric='accuracy')
    prefix = f'{backbone} {name}' if backbone else name
    safe_name = prefix.lower().replace(' ', '_').replace('-', '_').replace('(', '').replace(')', '')
    print(f'=== {prefix} ===')
    print(f'Blocks used: {df_compare.shape[0]}')
    print(f'Models used: {df_compare.shape[1]}')
    print(f'Model columns: {list(df_compare.columns)}')
    if df_compare.shape[0] < 5 or df_compare.shape[1] < 2:
        print('Autorank requires at least 5 paired rows and 2 models. Skipping this analysis.')
        return None

    df_compare.to_csv(OUTPUT_DIR / f'autorank_{safe_name}_compare_matrix.csv')
    result = autorank(df_compare, alpha=alpha, verbose=False, order='descending')
    print(result)

    report_path = OUTPUT_DIR / f'autorank_{safe_name}_report.txt'
    with report_path.open('w', encoding='utf-8') as f:
        f.write(str(result))
        f.write('\n')

    rankdf = getattr(result, 'rankdf', None)
    if rankdf is not None:
        rankdf.to_csv(OUTPUT_DIR / f'autorank_{safe_name}_rankdf.csv')
        display(rankdf)

    plt.figure(figsize=(12, 5))
    plot_stats(result, allow_insignificant=True)
    plt.tight_layout()
    fig_path = OUTPUT_DIR / f'autorank_{safe_name}_cd_diagram.png'
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'CD diagram saved to {fig_path}')
    print(f'Comparison matrix saved to {OUTPUT_DIR / f"autorank_{safe_name}_compare_matrix.csv"}')
    return result


In [42]:
autorank_results = {}
for backbone in ['mBERT', 'XLM-R']:
    autorank_results[(backbone, 'overall')] = run_autorank_analysis('Overall test splits', TEST_SPLITS, backbone=backbone)
    autorank_results[(backbone, 'monolingual')] = run_autorank_analysis('Monolingual test splits', MONOLINGUAL_SPLITS, backbone=backbone)
    autorank_results[(backbone, 'multilingual')] = run_autorank_analysis('Multilingual test splits', MULTILINGUAL_SPLITS, backbone=backbone)
    autorank_results[(backbone, 'cross_lingual')] = run_autorank_analysis('Cross-lingual test splits', CROSSLINGUAL_SPLITS, backbone=backbone)

=== mBERT Overall test splits ===
Blocks used: 135
Models used: 5
Model columns: ['mBERT (GloSeGAT)', 'mBERT (MarkCLS)', 'mBERT (PairCLS)', 'mBERT (TargetPOS)', 'mBERT (TargetSim)']
                   meanrank  median  mad ci_lower ci_upper effect_size   magnitude effect_size_above magnitude_above
mBERT (PairCLS)    4.766667    69.6  6.6     63.9     73.1         0.0  negligible               0.0      negligible
mBERT (MarkCLS)    3.500000    73.1  5.6     68.8     75.0   -0.385709       small         -0.385709           small
mBERT (TargetSim)  2.296296    79.3  2.8     76.8     80.7   -1.290569       large         -0.944582           large
mBERT (GloSeGAT)   2.288889    78.9  3.8     77.1     81.6   -1.164823       large          0.080834      negligible
mBERT (TargetPOS)  2.148148    79.3  2.3     77.6     80.3   -1.323825       large         -0.085899      negligible
RankResult(rankdf=
                   meanrank  median  mad ci_lower ci_upper effect_size   magnitude effect_size_ab

,meanrank,median,mad,ci_lower,ci_upper,effect_size,magnitude,effect_size_above,magnitude_above
mBERT (PairCLS),4.766667,69.6,6.6,63.9,73.1,0.0,negligible,0.0,negligible
mBERT (MarkCLS),3.500000,73.1,5.6,68.8,75.0,-0.385709,small,-0.385709,small
mBERT (TargetSim),2.296296,79.3,2.8,76.8,80.7,-1.290569,large,-0.944582,large
mBERT (GloSeGAT),2.288889,78.9,3.8,77.1,81.6,-1.164823,large,0.080834,negligible
mBERT (TargetPOS),2.148148,79.3,2.3,77.6,80.3,-1.323825,large,-0.085899,negligible


CD diagram saved to /Users/antonio/Desktop/repositorios/Experiments/analysis_outputs/autorank_mbert_overall_test_splits_cd_diagram.png
Comparison matrix saved to /Users/antonio/Desktop/repositorios/Experiments/analysis_outputs/autorank_mbert_overall_test_splits_compare_matrix.csv
=== mBERT Monolingual test splits ===
Blocks used: 15
Models used: 5
Model columns: ['mBERT (GloSeGAT)', 'mBERT (MarkCLS)', 'mBERT (PairCLS)', 'mBERT (TargetPOS)', 'mBERT (TargetSim)']
                   meanrank       mean       std   ci_lower   ci_upper effect_size   magnitude effect_size_above magnitude_above
mBERT (PairCLS)    4.933333  81.340000  0.671672  80.823741  81.856259         0.0  negligible               0.0      negligible
mBERT (MarkCLS)    3.233333  83.773333  1.043666  82.971154  84.575513   -2.772698       large         -2.772698           large
mBERT (GloSeGAT)   2.433333  84.886667  0.717004  84.335565  85.437769   -5.105263       large          -1.24345           large
mBERT (TargetPOS) 

/var/folders/6n/c3qqjd3522d80j_50lf558xc0000gn/T/ipykernel_59911/1153655761.py:55: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/var/folders/6n/c3qqjd3522d80j_50lf558xc0000gn/T/ipykernel_59911/1153655761.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,meanrank,mean,std,ci_lower,ci_upper,effect_size,magnitude,effect_size_above,magnitude_above
mBERT (PairCLS),4.933333,81.340000,0.671672,80.823741,81.856259,0.0,negligible,0.0,negligible
mBERT (MarkCLS),3.233333,83.773333,1.043666,82.971154,84.575513,-2.772698,large,-2.772698,large
mBERT (GloSeGAT),2.433333,84.886667,0.717004,84.335565,85.437769,-5.105263,large,-1.24345,large
mBERT (TargetPOS),2.433333,84.653333,1.521215,83.484101,85.822566,-2.817824,large,0.196217,negligible
mBERT (TargetSim),1.966667,85.153333,0.930335,84.438261,85.868405,-4.699827,large,-0.39655,small


...

/var/folders/6n/c3qqjd3522d80j_50lf558xc0000gn/T/ipykernel_59911/1153655761.py:55: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/var/folders/6n/c3qqjd3522d80j_50lf558xc0000gn/T/ipykernel_59911/1153655761.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


CD diagram saved to /Users/antonio/Desktop/repositorios/Experiments/analysis_outputs/autorank_xlm_r_multilingual_test_splits_cd_diagram.png
Comparison matrix saved to /Users/antonio/Desktop/repositorios/Experiments/analysis_outputs/autorank_xlm_r_multilingual_test_splits_compare_matrix.csv
=== XLM-R Cross-lingual test splits ===
Blocks used: 60
Models used: 5
Model columns: ['XLM-R (GloSeGAT)', 'XLM-R (MarkCLS)', 'XLM-R (PairCLS)', 'XLM-R (TargetPOS)', 'XLM-R (TargetSim)']
                   meanrank  median  mad ci_lower ci_upper effect_size   magnitude effect_size_above magnitude_above
XLM-R (PairCLS)    4.333333   70.90  2.5     69.3     74.0         0.0  negligible               0.0      negligible
XLM-R (TargetPOS)  3.466667   50.80  2.3     50.0     88.3    5.643963       large          5.643963           large
XLM-R (MarkCLS)    3.100000   82.90  0.9     82.1     83.8   -4.307942       large        -12.397414           large
XLM-R (GloSeGAT)   2.083333   87.20  2.0     84.2     

,meanrank,median,mad,ci_lower,ci_upper,effect_size,magnitude,effect_size_above,magnitude_above
XLM-R (PairCLS),4.333333,70.90,2.5,69.3,74.0,0.0,negligible,0.0,negligible
XLM-R (TargetPOS),3.466667,50.80,2.3,50.0,88.3,5.643963,large,5.643963,large
XLM-R (MarkCLS),3.100000,82.90,0.9,82.1,83.8,-4.307942,large,-12.397414,large
XLM-R (GloSeGAT),2.083333,87.20,2.0,84.2,88.5,-4.856425,large,-1.870195,large
XLM-R (TargetSim),2.016667,87.85,1.5,85.3,89.0,-5.545634,large,-0.248007,small


CD diagram saved to /Users/antonio/Desktop/repositorios/Experiments/analysis_outputs/autorank_xlm_r_cross_lingual_test_splits_cd_diagram.png
Comparison matrix saved to /Users/antonio/Desktop/repositorios/Experiments/analysis_outputs/autorank_xlm_r_cross_lingual_test_splits_compare_matrix.csv


/var/folders/6n/c3qqjd3522d80j_50lf558xc0000gn/T/ipykernel_59911/1153655761.py:55: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/var/folders/6n/c3qqjd3522d80j_50lf558xc0000gn/T/ipykernel_59911/1153655761.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Optional: Mean Accuracy by Dataset Type

This table averages each model over monolingual, multilingual, cross-lingual, and overall test splits. It is useful for a compact results table in the article.

In [43]:
type_rows = []
dataset_type_specs = [
    ('monolingual', MONOLINGUAL_SPLITS),
    ('multilingual', MULTILINGUAL_SPLITS),
    ('cross_lingual', CROSSLINGUAL_SPLITS),
    ('overall', TEST_SPLITS),
]
for model, group in metrics_df[metrics_df['split'].isin(TEST_SPLITS)].groupby('model'):
    meta = group[['backbone', 'variant']].iloc[0].to_dict()
    for split_type, splits in dataset_type_specs:
        sub = group[group['split'].isin(splits)]
        if sub.empty:
            continue
        type_rows.append({
            'model': model,
            **meta,
            'dataset_type': split_type,
            'mean_accuracy_across_splits_pct': 100 * sub['accuracy_mean'].mean(),
            'std_accuracy_across_splits_pct': 100 * sub['accuracy_mean'].std(ddof=1),
            'n_splits': sub['split'].nunique(),
        })

dataset_type_order = {name: i for i, (name, _) in enumerate(dataset_type_specs)}
dataset_type_summary = pd.DataFrame(type_rows)
dataset_type_summary['dataset_type_order'] = dataset_type_summary['dataset_type'].map(dataset_type_order)
dataset_type_summary = dataset_type_summary.sort_values(['dataset_type_order', 'mean_accuracy_across_splits_pct'], ascending=[True, False]).drop(columns='dataset_type_order')
display(dataset_type_summary)
dataset_type_summary.to_csv(OUTPUT_DIR / 'accuracy_by_dataset_type.csv', index=False)

,model,backbone,variant,dataset_type,mean_accuracy_across_splits_pct,std_accuracy_across_splits_pct,n_splits
4,XLM-R (MarkCLS),XLM-R,MarkCLS,monolingual,89.953333,NaN,1
0,XLM-R (GloSeGAT),XLM-R,GloSeGAT,monolingual,87.740000,NaN,1
16,XLM-R (TargetSim),XLM-R,TargetSim,monolingual,87.526667,NaN,1
36,mBERT (TargetSim),mBERT,TargetSim,monolingual,85.153333,NaN,1
20,mBERT (GloSeGAT),mBERT,GloSeGAT,monolingual,84.886667,NaN,1
32,mBERT (TargetPOS),mBERT,TargetPOS,monolingual,84.653333,NaN,1
24,mBERT (MarkCLS),mBERT,MarkCLS,monolingual,83.773333,NaN,1
8,XLM-R (PairCLS),XLM-R,PairCLS,monolingual,81.460000,NaN,1
28,mBERT (PairCLS),mBERT,PairCLS,monolingual,81.340000,NaN,1
12,XLM-R (TargetPOS),XLM-R,TargetPOS,monolingual,68.646667,NaN,1


## Combined Autorank Figure

This cell combines the six backbone-specific Critical Difference diagrams into one final image for inspection or article drafting.


In [44]:
from PIL import Image, ImageChops

combined_specs = [
    ('Overall', 'mBERT', OUTPUT_DIR / 'autorank_mbert_overall_test_splits_cd_diagram.png'),
    ('Overall', 'XLM-R', OUTPUT_DIR / 'autorank_xlm_r_overall_test_splits_cd_diagram.png'),
    ('Monolingual', 'mBERT', OUTPUT_DIR / 'autorank_mbert_monolingual_test_splits_cd_diagram.png'),
    ('Monolingual', 'XLM-R', OUTPUT_DIR / 'autorank_xlm_r_monolingual_test_splits_cd_diagram.png'),
    ('Multilingual', 'mBERT', OUTPUT_DIR / 'autorank_mbert_multilingual_test_splits_cd_diagram.png'),
    ('Multilingual', 'XLM-R', OUTPUT_DIR / 'autorank_xlm_r_multilingual_test_splits_cd_diagram.png'),
    ('Cross-lingual', 'mBERT', OUTPUT_DIR / 'autorank_mbert_cross_lingual_test_splits_cd_diagram.png'),
    ('Cross-lingual', 'XLM-R', OUTPUT_DIR / 'autorank_xlm_r_cross_lingual_test_splits_cd_diagram.png'),
]

missing = [str(path) for _, _, path in combined_specs if not path.exists()]
if missing:
    raise FileNotFoundError('Run the backbone-specific autorank cell first. Missing files: ' + ', '.join(missing))

def trim_whitespace(img, padding=20):
    img = img.convert('RGB')
    bg = Image.new('RGB', img.size, (255, 255, 255))
    diff = ImageChops.difference(img, bg)
    bbox = diff.getbbox()
    if bbox is None:
        return img
    left, upper, right, lower = bbox
    left = max(left - padding, 0)
    upper = max(upper - padding, 0)
    right = min(right + padding, img.size[0])
    lower = min(lower + padding, img.size[1])
    return img.crop((left, upper, right, lower))

fig, axes = plt.subplots(4, 2, figsize=(12, 8.6))
fig.subplots_adjust(left=0.02, right=0.98, top=0.965, bottom=0.025, wspace=0.04, hspace=0.04)

for ax, (split_group, backbone, path) in zip(axes.ravel(), combined_specs):
    img = trim_whitespace(Image.open(path), padding=5)
    ax.imshow(img)
    ax.set_title(f'{split_group} - {backbone}', fontsize=12, fontweight='bold', pad=1)
    ax.axis('off')

combined_path = OUTPUT_DIR / 'autorank_combined_cd_diagrams_vertical.png'
fig.savefig(combined_path, dpi=300, bbox_inches='tight', pad_inches=0.01)
plt.show()
print(f'Combined vertical autorank figure saved to {combined_path}')

Combined vertical autorank figure saved to /Users/antonio/Desktop/repositorios/Experiments/analysis_outputs/autorank_combined_cd_diagrams_vertical.png


/var/folders/6n/c3qqjd3522d80j_50lf558xc0000gn/T/ipykernel_59911/2368184996.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
